# Day 20. Next Token Prediction

**오늘의 목표**
- GPT가 학습하는 단 하나의 목표, "다음 Token 맞히기"가 무엇인지 설명할 수 있다
- 한 칸 밀린 `x` / `y` 데이터를 직접 만들고, 문장 하나에서 학습 예제가 **T개** 나온다는 걸 안다
- 가장 단순한 언어 모델(Bigram)을 학습시키고, **한 Token씩 생성**하는 과정을 직접 구현한다
- Training과 Inference가 어떻게 다른지 설명할 수 있다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(42)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()]

## 1. GPT의 목표는 딱 하나: 다음 Token 맞히기

```text
입력: 나는 오늘 학교
정답: 에
```

GPT는 학습하는 동안 **이것 하나만** 합니다. 번역, 요약, 코딩을 따로 배우지 않아요.

그런데 왜 이것만으로 똑똑해질까요? **다음 Token을 잘 맞히려면 많은 걸 알아야 하기 때문**입니다.

| 문장 | 다음 Token을 맞히려면 알아야 하는 것 |
|---|---|
| "나는 어제 학교에 ___" | 문법 (과거형 → "갔다") |
| "대한민국의 수도는 ___" | 사실 지식 ("서울") |
| "3 + 4 = ___" | 계산 ("7") |
| "철수는 배가 고파서 ___" | 상식, 인과관계 ("밥을") |

> 💡 "다음 Token 예측"은 단순해 보이지만, 정말 잘하려면 언어와 세상에 대한 이해가 필요합니다.

## 2. 학습 데이터 만들기: 한 칸 밀기

### 2-1. 커리큘럼 예시 (오타 수정)

커리큘럼 5-3에는 `y: 는 오늘 학교 에`라고 적혀 있는데, 한 칸 밀린 구조라면 이렇게 되어야 합니다.

```text
x: 나는  오늘  학교
y: 오늘  학교  에
```

`y`는 `x`를 **왼쪽으로 한 칸 민 것**입니다. 각 위치에서 "다음에 올 Token"이 정답이에요.

In [ ]:
tokens = ["나는", "오늘", "학교", "에"]

x = tokens[:-1]   # 마지막 하나 빼고
y = tokens[1:]    # 처음 하나 빼고

print("x:", x)
print("y:", y)
print()
for i in range(len(x)):
    print(f"{x[:i+1]} 다음에는 → '{y[i]}'")

**중요한 관찰**: Token 4개짜리 문장 하나에서 학습 문제가 **3개** 나왔습니다.
- "나는" 다음 → 오늘
- "나는 오늘" 다음 → 학교
- "나는 오늘 학교" 다음 → 에

문맥 길이가 1, 2, 3인 문제를 **한꺼번에** 학습합니다.

### 2-2. 글자 단위로 실제 데이터 만들기

Day 8에서 만든 글자 단위 Tokenizer로 작은 한국어 말뭉치를 숫자로 바꿔봅시다.

In [ ]:
text = (
    "나는 오늘 학교에 간다. 학교에서 친구를 만났다. 친구와 함께 점심을 먹었다. "
    "점심은 김밥이었다. 김밥은 맛있었다. 오후에는 도서관에 갔다. 도서관에서 책을 읽었다. "
    "책은 재미있었다. 저녁에는 집에 왔다. 집에서 엄마가 밥을 해주셨다. 밥을 먹고 숙제를 했다. "
    "숙제는 어려웠다. 그래도 끝까지 했다. 오늘은 좋은 하루였다. 내일도 학교에 간다. "
    "내일은 친구와 운동장에서 축구를 할 것이다. 축구는 재미있다. 나는 축구를 좋아한다. "
    "친구도 축구를 좋아한다. 우리는 매일 학교에 간다. 학교는 즐겁다. 나는 오늘도 공부를 했다."
)

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
print("글자 수       :", len(text))
print("vocab_size   :", vocab_size)
print("앞 20개 ID    :", data[:20].tolist())
print("decode 확인   :", decode(data[:20].tolist()))

### 2-3. 길이 T짜리 조각 하나 = 학습 예제 T개

`block_size`(문맥 최대 길이)만큼 잘라서 `x`, 한 칸 밀어서 `y`를 만듭니다.

In [ ]:
block_size = 8

x = data[:block_size]
y = data[1:block_size + 1]

print("x:", x.tolist(), "→", repr(decode(x.tolist())))
print("y:", y.tolist(), "→", repr(decode(y.tolist())))
print()
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"문맥 {repr(decode(context.tolist())):12s} → 정답 {repr(itos[target.item()])}")

> 💡 길이 8짜리 조각 하나에서 학습 예제가 **8개** 나옵니다.
> 이게 가능한 이유는 Day 14의 **Causal Mask** 덕분이에요. 위치 t는 0~t까지만 볼 수 있으니, 모든 위치를 한 번에 계산해도 커닝이 없습니다.

### 2-4. 배치로 뽑기: `get_batch`

말뭉치의 **랜덤 위치**에서 `B`개 조각을 뽑아 `(B, T)` 텐서를 만듭니다. nanoGPT와 같은 방식이에요.

In [ ]:
def get_batch(batch_size, block_size):
    ix = torch.randint(len(data) - block_size, (batch_size,))        # 랜덤 시작 위치 B개
    x = torch.stack([data[i : i + block_size] for i in ix])          # (B, T)
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])  # (B, T), 한 칸 밀림
    return x, y

xb, yb = get_batch(batch_size=4, block_size=8)
print("xb:", tuple(xb.shape))
print("yb:", tuple(yb.shape))
print()
for b in range(4):
    print(f"x[{b}] {repr(decode(xb[b].tolist())):14s}   y[{b}] {repr(decode(yb[b].tolist()))}")

`B=4`, `T=8`이면 이 배치 하나에 학습 예제가 4 × 8 = **32개** 들어 있습니다.

## 3. 가장 단순한 언어 모델: Bigram

**Bigram** = "바로 앞 글자 하나만 보고 다음 글자를 맞히는" 모델입니다.

구조는 `nn.Embedding(vocab_size, vocab_size)` 하나뿐이에요.
- Day 9에서 Embedding은 "ID로 표에서 행 하나 꺼내기"라고 배웠죠?
- 여기서는 꺼낸 행(길이 `vocab_size`)을 **바로 다음 글자의 점수(logits)** 로 씁니다.

```text
"학" 의 ID  →  표에서 "학" 행 꺼내기  →  [각 글자가 다음에 올 점수 vocab_size개]
                                         "교" 점수가 높아지도록 학습
```

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_table(idx)            # (B, T) → (B, T, vocab_size)

        if targets is None:
            return logits, None

        B, T, V = logits.shape
        loss = F.cross_entropy(logits.view(B * T, V), targets.view(B * T))   # Day 6
        return logits, loss

model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print("입력 xb  :", tuple(xb.shape))
print("logits   :", tuple(logits.shape), " ← 위치마다 다음 글자 점수")
print("loss     :", round(loss.item(), 3))
print("log(V)   :", round(math.log(vocab_size), 3), " ← 아무것도 모르는 모델의 loss (Day 6)")

학습 전 loss는 `log(vocab_size)`와 비슷하거나 **조금 더 큽니다**.
- 모든 글자에 확률을 똑같이 주면 정확히 `log(vocab_size)`가 나와요. (Day 6)
- `nn.Embedding`의 초기값은 랜덤이라 점수가 살짝 들쭉날쭉해서, 틀린 글자에 괜히 자신 있는 경우가 생겨 조금 더 높게 나옵니다.

어느 쪽이든 "아직 아무것도 모른다"는 뜻이에요. 학습을 시작하면 이 숫자부터 내려갑니다.

> 💡 `logits.view(B * T, V)` / `targets.view(B * T)`: `F.cross_entropy`는 `(N, V)`와 `(N,)`을 받기 때문에 B와 T 축을 하나로 합칩니다. (Day 6 7절)

## 4. 학습시키기

Day 7에서 배운 5줄 학습 루프 그대로입니다.

In [ ]:
model = BigramLanguageModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

losses = []
for step in range(2000):
    xb, yb = get_batch(batch_size=32, block_size=8)
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 400 == 0:
        print(f"step {step:4d}  loss {loss.item():.3f}")

print(f"최종 loss: {sum(losses[-100:]) / 100:.3f}   (시작: 약 {math.log(vocab_size):.3f})")

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(losses, alpha=0.4, label="step별 loss")
smooth = torch.tensor(losses).unfold(0, 50, 1).mean(dim=1)
plt.plot(range(49, len(losses)), smooth, linewidth=2.5, label="50 step 평균")
plt.axhline(math.log(vocab_size), color="gray", linestyle="--", label=f"log(V) = {math.log(vocab_size):.2f}")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Bigram 모델 학습 곡선", fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 4-1. 무엇을 배웠나 들여다보기

"학" 다음에 어떤 글자가 올 확률이 높다고 배웠는지 확인해봅시다.

In [ ]:
def top_next(ch, k=5):
    with torch.no_grad():
        logits = model.token_table.weight[stoi[ch]]
        probs = F.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    return [(itos[i.item()], round(p.item() * 100, 1)) for p, i in zip(top.values, top.indices)]

for ch in ["학", "친", "축", "다"]:
    print(f"'{ch}' 다음 →", top_next(ch))

"학" 다음엔 "교", "축" 다음엔 "구"처럼 말뭉치에서 자주 붙어 나온 글자를 배웠습니다.
`"다"` 다음은 `"."`나 공백이 많겠죠.

## 5. 생성하기: 한 Token씩 이어 붙이기 (Autoregressive)

커리큘럼 5-4의 그림입니다.

```text
나는
 ↓
나는 오늘
 ↓
나는 오늘 학교에
 ↓
나는 오늘 학교에 갔다
```

한 번에 Token **하나**를 만들고, 그걸 입력에 **붙여서** 다시 넣는 걸 반복합니다.

### 5-1. 한 스텝씩 따라가기

In [ ]:
torch.manual_seed(0)
idx = torch.tensor([encode("나는")])    # (B=1, T=2)
print("시작:", repr(decode(idx[0].tolist())))

for step in range(6):
    with torch.no_grad():
        logits, _ = model(idx)                       # (1, T, V)
    last = logits[:, -1, :]                          # (1, V)  마지막 위치만! (Day 1 Q4)
    probs = F.softmax(last, dim=-1)                  # (1, V)  확률로 (Day 5)
    next_id = torch.multinomial(probs, num_samples=1)  # (1, 1)  확률대로 하나 뽑기
    idx = torch.cat([idx, next_id], dim=1)           # (1, T+1) 뒤에 붙이기

    ch = itos[next_id.item()]
    p = probs[0, next_id.item()].item()
    print(f"step {step+1}: '{ch}' 선택 (확률 {p*100:4.1f}%)  →  {repr(decode(idx[0].tolist()))}")

### 5-2. `generate` 함수로 정리하기

nanoGPT의 `generate`와 같은 구조입니다. `block_size`보다 길어지면 **최근 `block_size`개만** 잘라서 넣어요.
(Bigram은 마지막 글자만 쓰지만, 진짜 GPT는 위치 임베딩 크기 때문에 반드시 잘라야 합니다. Day 10)

In [ ]:
@torch.no_grad()
def generate(model, idx, max_new_tokens, block_size=8, temperature=1.0, greedy=False):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]              # 1) 최근 block_size개만
        logits, _ = model(idx_cond)                  # 2) 모델에 넣기
        logits = logits[:, -1, :] / temperature      # 3) 마지막 위치 + Temperature (Day 5)
        probs = F.softmax(logits, dim=-1)            # 4) 확률로
        if greedy:
            next_id = torch.argmax(probs, dim=-1, keepdim=True)
        else:
            next_id = torch.multinomial(probs, num_samples=1)  # 5) 하나 뽑기
        idx = torch.cat([idx, next_id], dim=1)       # 6) 붙이기
    return idx

### 5-3. 학습 전 vs 학습 후

In [ ]:
torch.manual_seed(1)
untrained = BigramLanguageModel(vocab_size)
start = torch.tensor([encode("나는")])

print("학습 전:", decode(generate(untrained, start, 60)[0].tolist()))
print()
print("학습 후:", decode(generate(model, start, 60)[0].tolist()))

학습 전은 완전히 랜덤한 글자 나열이고, 학습 후는 **글자 단위로는 그럴듯하지만 의미는 엉망**입니다.
"학교", "축구" 같은 짝은 맞추지만, 문장 전체는 말이 안 되죠. 바로 앞 글자 하나만 보기 때문이에요.

### 5-4. Temperature와 Greedy (Day 5 복습)

In [ ]:
for t in [0.5, 1.0, 1.5]:
    torch.manual_seed(2)
    print(f"temperature {t}:", decode(generate(model, start, 50, temperature=t)[0].tolist()))
print()
print("greedy       :", decode(generate(model, start, 50, greedy=True)[0].tolist()))

- Temperature가 낮으면 자주 나온 패턴만 반복하고, 높으면 엉뚱한 글자가 섞입니다.
- Greedy는 매번 1등만 고르기 때문에 금방 **같은 패턴을 무한 반복**하는 경우가 많아요. 그래서 실제 LLM은 보통 Sampling을 씁니다.

## 6. Training vs Inference

커리큘럼 5-5를 오늘 코드와 연결해봅시다.

### Training (오늘 4절)
```text
Input (B, T)           ← 정답 y도 같이 있음
 ↓
Prediction (B, T, V)   ← T개 위치를 한 번에 계산 (Causal Mask 덕분)
 ↓
Target과 비교          ← cross_entropy(logits, y)
 ↓
Loss
 ↓
Backpropagation        ← Weight 수정
```

### Inference (오늘 5절)
```text
Input (1, T)           ← 정답 없음
 ↓
Next Token Prediction  ← 마지막 위치의 logits만 사용
 ↓
Token 선택             ← softmax → multinomial (또는 argmax)
 ↓
Input에 추가           ← torch.cat
 ↓
반복                   ← max_new_tokens번
```

| | Training | Inference |
|---|---|---|
| 정답(y) | 있음 | 없음 |
| 한 번에 처리 | T개 위치 **동시에** | Token **하나씩** |
| 쓰는 logits | 모든 위치 `(B, T, V)` | 마지막 위치 `[:, -1, :]` |
| Loss / backward | 계산함 | 안 함 (`torch.no_grad()`) |
| Weight | 바뀜 | 고정 |
| 속도 | 병렬이라 빠름 | 순차라 느림 |

> 💡 Inference가 느린 이유: Token 하나 만들 때마다 모델 전체를 다시 돌립니다.
> 이미 계산한 앞부분의 K, V를 저장해서 재사용하는 **KV Cache**라는 기법으로 빠르게 만드는데, 추가 학습 단계에서 다룹니다.

## 7. LLM과의 연결: Bigram의 한계 → GPT

Bigram은 **바로 앞 글자 하나**만 봅니다.

```text
"나는 오늘 학교에 ___"
Bigram이 보는 것:            "에"   (공백 직전 글자 하나)
GPT가 보는 것:   "나는 오늘 학교에"   (Attention으로 앞 문맥 전체)
```

오늘 만든 것과 Day 19의 GPT 구조를 비교하면:

| | Bigram (오늘) | mini GPT (Day 19, 28) |
|---|---|---|
| 모델 | Embedding 표 하나 | Embedding + Position + Block × N + lm_head |
| 문맥 | 직전 Token 1개 | 최대 block_size개 |
| 데이터 x / y | 한 칸 밀기 | **똑같음** |
| Loss | cross_entropy | **똑같음** |
| 학습 루프 | AdamW 5줄 | **똑같음** |
| generate | 마지막 logits → softmax → 뽑기 → 붙이기 | **똑같음** |

> 💡 오늘 만든 `get_batch`, 학습 루프, `generate`는 Day 28 mini GPT에서 **거의 그대로** 씁니다.
> 바뀌는 건 `BigramLanguageModel`을 GPT 모델로 갈아끼우는 것뿐이에요.

## 8. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** Token ID 시퀀스 `[5, 2, 8, 1, 7]`에서 `block_size = 4`로 `x`, `y`를 만드세요. (맨 앞에서부터)

In [ ]:
seq = [5, 2, 8, 1, 7]

x = "?"   # TODO: 리스트로
y = "?"   # TODO: 리스트로

assert isinstance(x, list) and isinstance(y, list), "리스트로 적어주세요"
assert len(x) == 4 and len(y) == 4, "block_size만큼의 길이여야 해요"
assert x == seq[:4] and y == seq[1:5], "y는 x를 어떻게 민 것인지 다시 생각해보세요"
print("Q1 정답!")

**Q2.** `get_batch(batch_size=16, block_size=32)`로 뽑은 배치 하나에는 "문맥 → 정답" 학습 예제가 몇 개 들어 있을까요?

In [ ]:
my_answer = "?"   # TODO: 숫자 하나

xb, yb = get_batch(batch_size=16, block_size=32)
assert my_answer == xb.numel(), "조각 하나에서 예제가 몇 개 나오는지(2-3절) 떠올려보세요"
print("Q2 정답!")

**Q3.** `generate`를 쓰지 않고, **Greedy로 한 스텝**만 생성하는 함수를 완성하세요.
입력 `idx`는 `(1, T)`이고, 결과는 다음 Token 하나가 붙은 `(1, T+1)`이어야 합니다.

힌트: 마지막 위치의 logits → `argmax` → `torch.cat`

In [ ]:
@torch.no_grad()
def greedy_one_step(model, idx):
    return None  # TODO

idx = torch.tensor([encode("학")])
out = greedy_one_step(model, idx)

assert out is not None, "함수를 완성해주세요"
assert tuple(out.shape) == (1, 2), "결과 shape를 확인해보세요"
assert torch.equal(out, generate(model, idx, 1, greedy=True)), "마지막 위치의 logits를 썼는지, argmax를 썼는지 확인해보세요"
print("Q3 정답!", repr(decode(out[0].tolist())))

**Q4.** vocab_size가 **100**인 언어 모델을 처음 만들었을 때(학습 전), loss는 대략 얼마일까요?
소수점 둘째 자리까지 적으세요. (Day 6)

In [ ]:
my_loss = "?"   # TODO: 숫자 하나 (예: 1.23)

assert isinstance(my_loss, (int, float)), "숫자로 적어주세요"
torch.manual_seed(0)
probe = BigramLanguageModel(100)
probe.token_table.weight.data.mul_(0.01)       # 점수를 거의 같게 → '아무것도 모르는' 모델
ids = torch.randint(100, (8, 16))
_, actual = probe(ids, torch.randint(100, (8, 16)))
assert abs(my_loss - actual.item()) < 0.01, "아무것도 모르는 모델은 모든 Token에 확률을 똑같이 줍니다. 그때의 -log(p)는?"
print(f"Q4 정답! 실제 초기 loss: {actual.item():.3f}")

## 9. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. GPT는 왜 "다음 Token 예측"만으로 다양한 능력을 갖게 되는가?
2. `x`와 `y`는 어떤 관계인가? 길이 T인 조각 하나에서 학습 예제가 몇 개 나오는가?
3. 생성할 때 왜 `logits[:, -1, :]`(마지막 위치)만 쓰는가?
4. Training과 Inference는 무엇이 다른가? (정답 유무, 병렬/순차, backward)
5. Bigram 모델의 한계는 무엇이고, GPT는 이를 어떻게 해결하는가?

**내 답:**

1.
2.
3.
4.
5.